# Step E — Pen-and-paper verification via PTDF




## Imports

In [1]:
from pathlib import Path
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


c:\Users\terry\miniforge3\envs\IEG\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## Rebuild the Step D network (silent)

This block reproduces the Step D setup exactly, then optimises. After this, `net` has
the same flows and capacities as at the end of Step D.


In [2]:
# ----- Cost table (same as Step A) -----
data = {
    "capital_cost": [
        1500000/25 + 60000,
        800000/25 + 14000,
        700000/25 + 24000,
    ],
    "marginal_cost": [0.0, 0.0, 9.5 * 3.6 / 0.56 + 2.30]
}
costs = pd.DataFrame(data, index=["wind_combined", "solar", "CCGT"])

# ----- Resolve paths -----
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "STEP D - electricity_demand.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent
cf_base = PROJECT_DIR / "Other country data" / "Energy charts data"

# ----- Denmark CFs + snapshot reference -----
dataframe_dk = pd.read_csv(PROJECT_DIR / "DK_2015_merged.csv", index_col=0, sep=",", parse_dates=True)
dataframe_dk.index = pd.to_datetime(dataframe_dk.index, utc=True).tz_localize(None)
dataframe_dk = dataframe_dk.sort_index()
CF_wind  = dataframe_dk["wind_cf_Unnamed: 1"].astype(float)
CF_solar = dataframe_dk["pv_cf_Unnamed: 1"].astype(float)
snapshots = dataframe_dk.index

# ----- Multi-country demand -----
demand_all = pd.read_csv(PROJECT_DIR / "STEP D - electricity_demand.csv",
                         sep=";", index_col=0, parse_dates=True)
demand_all.index = pd.to_datetime(demand_all.index, utc=True).tz_localize(None)
demand_all = demand_all.sort_index()
demand_2015 = demand_all[demand_all.index.year == 2015].copy()
demand_dk = demand_2015["DNK"].astype(float).reindex(snapshots)
demand_de = demand_2015["DEU"].astype(float).reindex(snapshots)
demand_se = demand_2015["SWE"].astype(float).reindex(snapshots)
demand_no = demand_2015["NOR"].astype(float).reindex(snapshots)

# ----- Neighbour CFs -----
cf_de_raw = pd.read_csv(cf_base / "Germany" / "Germany_hourly_capacity_factors.csv")
cf_de_raw["timestamp"] = pd.to_datetime(cf_de_raw["timestamp"])
cf_de_raw = cf_de_raw.set_index("timestamp").sort_index()
cf_de_raw.index = cf_de_raw.index.tz_localize(None)
cf_de = cf_de_raw.resample("h").mean()
cf_de["wind_combined"] = cf_de["Wind onshore"]
cf_de["solar"] = cf_de["Solar AC"]
cf_de["CCGT"]  = cf_de["Fossil gas"]
cf_de["nuclear"] = cf_de["Nuclear"]
cf_de = cf_de[["wind_combined", "solar", "CCGT", "nuclear"]].reindex(snapshots)

cf_se = pd.read_csv(cf_base / "Sweden" / "Sweden_hourly_capacity_factors.csv")
cf_se["timestamp"] = pd.to_datetime(cf_se["timestamp"])
cf_se = cf_se.set_index("timestamp").sort_index()
cf_se.index = cf_se.index.tz_localize(None)
cf_se["wind_combined"] = cf_se["Wind onshore"]
cf_se["nuclear"] = cf_se["Nuclear"]
cf_se["hydro"]   = cf_se["Hydro water reservoir"]
cf_se = cf_se[["wind_combined", "nuclear", "hydro"]].reindex(snapshots).bfill()

cf_no = pd.read_csv(cf_base / "Norway" / "Norway_hourly_capacity_factors.csv")
cf_no["timestamp"] = pd.to_datetime(cf_no["timestamp"])
cf_no = cf_no.set_index("timestamp").sort_index()
cf_no.index = cf_no.index.tz_localize(None)
cf_no["wind_combined"] = cf_no["Wind onshore"]
cf_no["hydro"] = cf_no["Hydro water reservoir"]
cf_no = cf_no[["wind_combined", "hydro"]].reindex(snapshots)

# ----- Build the network -----
net = pypsa.Network()
net.set_snapshots(snapshots)
for name, (x, y) in {"Denmark": (10.0, 56.0), "Germany": (10.5, 51.5),
                     "Sweden": (15.0, 59.5), "Norway": (10.0, 62.0)}.items():
    net.add("Bus", name, x=x, y=y)

net.add("Load", "load_DK", bus="Denmark", p_set=demand_dk.values)
net.add("Load", "load_DE", bus="Germany", p_set=demand_de.values)
net.add("Load", "load_SE", bus="Sweden",  p_set=demand_se.values)
net.add("Load", "load_NO", bus="Norway",  p_set=demand_no.values)

net.add("Carrier", ["wind_onshore", "solar", "CCGT",
                    "hydro", "nuclear", "coal", "battery"],
        color=["blue", "yellow", "brown", "cyan", "purple", "grey", "purple"])

# Denmark — extendable
net.add("Generator", "DK_wind", bus="Denmark", carrier="wind_combined",
        capital_cost=costs.loc["wind_combined", "capital_cost"],
        marginal_cost=costs.loc["wind_combined", "marginal_cost"],
        p_max_pu=CF_wind.values, p_nom_extendable=True)
net.add("Generator", "DK_solar", bus="Denmark", carrier="solar",
        capital_cost=costs.loc["solar", "capital_cost"],
        marginal_cost=costs.loc["solar", "marginal_cost"],
        p_max_pu=CF_solar.values, p_nom_extendable=True)
net.add("Generator", "DK_CCGT", bus="Denmark", carrier="CCGT",
        capital_cost=costs.loc["CCGT", "capital_cost"],
        marginal_cost=costs.loc["CCGT", "marginal_cost"],
        efficiency=0.58, p_nom_extendable=True)

# Germany — fixed
net.add("Generator", "DE_wind", bus="Germany", carrier="wind_combined",
        p_nom=41300, marginal_cost=0, p_max_pu=cf_de["wind_combined"].values,
        p_nom_extendable=False)
net.add("Generator", "DE_solar", bus="Germany", carrier="solar",
        p_nom=37000, marginal_cost=0, p_max_pu=cf_de["solar"].values,
        p_nom_extendable=False)
net.add("Generator", "DE_CCGT", bus="Germany", carrier="CCGT",
        p_nom=28360, marginal_cost=60.0, p_nom_extendable=False)
net.add("Generator", "DE_nuclear", bus="Germany", carrier="nuclear",
        p_nom=10800, marginal_cost=10.0, p_nom_extendable=False)
net.add("Generator", "DE_coal", bus="Germany", carrier="coal",
        p_nom=21420, marginal_cost=30.0, p_nom_extendable=False)

# Sweden — fixed
net.add("Generator", "SE_hydro", bus="Sweden", carrier="hydro",
        p_nom=15920, marginal_cost=5.0, p_max_pu=cf_se["hydro"].values,
        p_nom_extendable=False)
net.add("Generator", "SE_nuclear", bus="Sweden", carrier="nuclear",
        p_nom=8900, marginal_cost=10.0, p_nom_extendable=False)
net.add("Generator", "SE_wind", bus="Sweden", carrier="wind_combined",
        p_nom=5500, marginal_cost=0, p_max_pu=cf_se["wind_combined"].values,
        p_nom_extendable=False)

# Norway — fixed
net.add("Generator", "NO_hydro", bus="Norway", carrier="hydro",
        p_nom=29900, marginal_cost=5.0, p_max_pu=cf_no["hydro"].values,
        p_nom_extendable=False)
net.add("Generator", "NO_wind", bus="Norway", carrier="wind_combined",
        p_nom=700, marginal_cost=0, p_max_pu=cf_no["wind_combined"].values,
        p_nom_extendable=False)

# Transmission lines (400 kV, x = 0.1 pu)
for bus in net.buses.index:
    net.buses.loc[bus, "v_nom"] = 380
for name, bus0, bus1, s_nom, length in [
    ("line_DK_DE", "Denmark", "Germany", 3500, 360),
    ("line_DK_SE", "Denmark", "Sweden",  1700, 520),
    ("line_DK_NO", "Denmark", "Norway",  1050, 570),
    ("line_SE_NO", "Sweden",  "Norway",  3500, 480),
    ("line_DE_SE", "Germany", "Sweden",   600, 820),
]:
    net.add("Line", name, bus0=bus0, bus1=bus1,
            s_nom=s_nom, x=0.1, r=0.01, length=length, p_nom=s_nom)

# Batteries (2024 Li-ion costs, same as Step C/D)
b_cap = 100_000 / 20 + 12_500 + (150_000 / 20) * 4   # 47,500 $/MW/year
for country, bus in [("DK", "Denmark"), ("DE", "Germany"),
                     ("SE", "Sweden"),  ("NO", "Norway")]:
    net.add("StorageUnit", f"{country}_battery", bus=bus, carrier="battery",
            capital_cost=b_cap, marginal_cost=0,
            efficiency_store=0.90**0.5, efficiency_dispatch=0.90**0.5,
            max_hours=4, cyclic_state_of_charge=True, p_nom_extendable=True)

# Optimise
net.optimize(solver_name="gurobi", solver_options={"output_flag": False})
print(f"Step D rebuilt — system cost: {net.objective/1e9:.3f} B$/y")


C:\Users\terry\AppData\Local\Temp\ipykernel_23348\1968716118.py:151: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  net.optimize(solver_name="gurobi", solver_options={"output_flag": False})
Index(['Denmark', 'Germany', 'Sweden', 'Norway'], dtype='str', name='name')
Index(['DK_wind', 'DE_wind', 'SE_wind', 'NO_wind'], dtype='str', name='name')
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-12-31 23:00:00'], dtype='datetime64[us]', name='snapshot', freq=None)
DatetimeIndex(['2015-10-25 01:00:00', '2015-12-31 23:

Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2773895


INFO:gurobipy:Set parameter LicenseID to value 2773895


...

Reading time = 1.44 seconds


INFO:gurobipy:Reading time = 1.44 seconds


obj: 613207 rows, 262807 columns, 1116871 nonzeros


INFO:gurobipy:obj: 613207 rows, 262807 columns, 1116871 nonzeros
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 262807 primals, 613207 duals
Objective: 2.14e+10
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Line-fix-s-lower, Line-fix-s-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, Kirchhoff-Voltage-Law, StorageUnit-energy_balance were not assigned to the network.


Step D rebuilt — system cost: 21.382 B$/y


## Step E — PTDF Verification of Line Flows

**Goal:** Using only the network topology and reactances, reproduce the line flows
that PyPSA computed for the first time step (2015-01-01 00:00).

### Notation

| Symbol | Meaning |
|--------|---------|
| $n$ | Number of buses (4: DK, DE, SE, NO) |
| $m$ | Number of lines (5: DK–DE, DK–SE, DK–NO, SE–NO, DE–SE) |
| $\boldsymbol{\theta}$ | Vector of bus voltage angles $(n \times 1)$ |
| $\mathbf{f}$ | Vector of active power flows on lines $(m \times 1)$ |
| $\mathbf{K}$ | Incidence matrix $(n \times m)$: $K_{il}=+1$ from end, $-1$ to end |
| $\mathbf{B}$ | Diagonal susceptance matrix $(m \times m)$: $B_{ll}=1/x_l$ |
| $\mathbf{L}$ | Weighted Laplacian $(n \times n)$: $\mathbf{L} = \mathbf{K}\mathbf{B}\mathbf{K}^\top$ |
| $\mathbf{L}_r$ | Reduced Laplacian: $\mathbf{L}$ with slack row and column removed |
| $\mathbf{p}$ | Nodal injections (generation + storage dispatch $-$ demand) |
| PTDF | $\mathbf{B}\,\mathbf{K}_r^\top\,\mathbf{L}_r^{-1}$, dimensions $(m \times (n-1))$ |

### Method

Under the DC approximation, the power flow on line $l$ is:

$$f_l = \frac{1}{x_l}\left(\theta_{\text{from}(l)} - \theta_{\text{to}(l)}\right)
\quad \Rightarrow \quad \mathbf{f} = \mathbf{B}\,\mathbf{K}^\top\,\boldsymbol{\theta}$$

The nodal balance imposes that the net injection at each bus equals the sum of
flows on its connected lines:

$$p_i = \sum_l K_{il} f_l \quad \Rightarrow \quad \mathbf{p} = \mathbf{K}\,\mathbf{f}$$

Substituting the flow equation into the nodal balance gives:

$$\mathbf{p} = \underbrace{\mathbf{K}\,\mathbf{B}\,\mathbf{K}^\top}_{\mathbf{L}}\,\boldsymbol{\theta}$$

The goal is to isolate $\boldsymbol{\theta}$ and substitute back into
$\mathbf{f} = \mathbf{B}\,\mathbf{K}^\top\,\boldsymbol{\theta}$, eliminating
the voltage angles entirely and expressing line flows directly as a function
of nodal injections.

However, $\mathbf{L}$ is singular (its rows sum to zero) and cannot be inverted.
We fix Denmark as the slack bus ($\theta_0 = 0$) and remove its row and column
to obtain the reduced, invertible Laplacian $\mathbf{L}_r$:

$$\boldsymbol{\theta}_r = \mathbf{L}_r^{-1}\,\mathbf{p}_r$$

Substituting back into the line flow equation yields the PTDF formulation:

$$\mathbf{f} = \underbrace{\mathbf{B}\,\mathbf{K}_r^\top\,\mathbf{L}_r^{-1}}_{\text{PTDF}}\,\mathbf{p}_r$$

Each element $\text{PTDF}_{l,i}$ represents the fraction of 1 MW injected at
bus $i$ that flows through line $l$.

### Implementation steps
1. Build the incidence matrix $\mathbf{K}$ from the network topology
2. Build the susceptance matrix $\mathbf{B}$ from line reactances
3. Compute the weighted Laplacian $\mathbf{L} = \mathbf{K}\mathbf{B}\mathbf{K}^\top$
4. Reduce $\mathbf{L}$ by removing the slack bus row and column → $\mathbf{L}_r$
5. Compute PTDF $= \mathbf{B}\,\mathbf{K}_r^\top\,\mathbf{L}_r^{-1}$
6. Read nodal injections $\mathbf{p}_r$ from PyPSA at $t_0$
7. Compute $\mathbf{f} = \text{PTDF}\,\mathbf{p}_r$ and verify against PyPSA

In [9]:
# --- E1: Incidence matrix K ---

import numpy as np
import pandas as pd

bus_order  = ["Denmark", "Germany", "Sweden", "Norway"]
line_order = ["DK-DE", "DK-SE", "DK-NO", "SE-NO", "DE-SE"]

# K[i, l] = +1 if bus i is the from-end of line l
#           -1 if bus i is the to-end of line l
#            0 otherwise
K = np.array([
    [ 1,  1,  1,  0,  0],   # Denmark
    [-1,  0,  0,  0,  1],   # Germany
    [ 0, -1,  0,  1, -1],   # Sweden
    [ 0,  0, -1, -1,  0],   # Norway
], dtype=float)

print("Incidence matrix K  (rows = buses, cols = lines):")
print(pd.DataFrame(K, index=bus_order, columns=line_order).to_string())
print(f"\nShape: {K.shape}  →  ({len(bus_order)} buses × {len(line_order)} lines)")

Incidence matrix K  (rows = buses, cols = lines):
         DK-DE  DK-SE  DK-NO  SE-NO  DE-SE
Denmark    1.0    1.0    1.0    0.0    0.0
Germany   -1.0    0.0    0.0    0.0    1.0
Sweden     0.0   -1.0    0.0    1.0   -1.0
Norway     0.0    0.0   -1.0   -1.0    0.0

Shape: (4, 5)  →  (4 buses × 5 lines)


In [10]:
# --- E2: Susceptance matrix B and weighted Laplacian L ---

# All lines have reactance x = 0.1 pu → susceptance b = 1/x = 10 pu
x = net.lines["x"].values   # reactance of each line
b = 1.0 / x                 # susceptance of each line

# Diagonal susceptance matrix B (m × m)
B = np.diag(b)

# Weighted Laplacian L = K B K^T  (n × n)
L = K @ B @ K.T

print("Line susceptances b = 1/x:")
print(pd.DataFrame({"line": line_order, "x [pu]": x, "b [pu]": b}).to_string(index=False))

print("\nWeighted Laplacian L  (rows/cols = buses):")
print(pd.DataFrame(L, index=bus_order, columns=bus_order).to_string())

print("\nVerification — row sums of L (must all be zero):")
print(L.sum(axis=1).round(10))

Line susceptances b = 1/x:
 line  x [pu]  b [pu]
DK-DE     0.1    10.0
DK-SE     0.1    10.0
DK-NO     0.1    10.0
SE-NO     0.1    10.0
DE-SE     0.1    10.0

Weighted Laplacian L  (rows/cols = buses):
         Denmark  Germany  Sweden  Norway
Denmark     30.0    -10.0   -10.0   -10.0
Germany    -10.0     20.0   -10.0     0.0
Sweden     -10.0    -10.0    30.0   -10.0
Norway     -10.0      0.0   -10.0    20.0

Verification — row sums of L (must all be zero):
[0. 0. 0. 0.]


In [11]:
# --- E3: Reduced Laplacian L_r and PTDF matrix ---

slack_idx = 0  # Denmark = slack bus (θ_0 = 0)
non_slack = [b for b in bus_order if b != bus_order[slack_idx]]

# Reduced Laplacian L_r: remove slack row and column  (n-1) × (n-1)
L_r = np.delete(np.delete(L, slack_idx, axis=0), slack_idx, axis=1)

# Reduced incidence matrix K_r^T: remove slack column  (m × (n-1))
KT_r = np.delete(K.T, slack_idx, axis=1)

# PTDF = B · K_r^T · L_r^{-1}  (m × (n-1))
L_r_inv = np.linalg.inv(L_r)
PTDF = B @ KT_r @ L_r_inv

print("Reduced Laplacian L_r  (rows/cols = non-slack buses):")
print(pd.DataFrame(L_r, index=non_slack, columns=non_slack).to_string())

print("\nL_r inverse:")
print(pd.DataFrame(L_r_inv, index=non_slack, columns=non_slack).round(6).to_string())

print("\nPTDF matrix  (rows = lines, cols = non-slack buses):")
print(pd.DataFrame(PTDF, index=line_order, columns=non_slack).round(4).to_string())

Reduced Laplacian L_r  (rows/cols = non-slack buses):
         Germany  Sweden  Norway
Germany     20.0   -10.0     0.0
Sweden     -10.0    30.0   -10.0
Norway       0.0   -10.0    20.0

L_r inverse:
         Germany  Sweden  Norway
Germany   0.0625   0.025  0.0125
Sweden    0.0250   0.050  0.0250
Norway    0.0125   0.025  0.0625

PTDF matrix  (rows = lines, cols = non-slack buses):
       Germany  Sweden  Norway
DK-DE   -0.625   -0.25  -0.125
DK-SE   -0.250   -0.50  -0.250
DK-NO   -0.125   -0.25  -0.625
SE-NO    0.125    0.25  -0.375
DE-SE    0.375   -0.25  -0.125


In [14]:
# --- E4: Nodal injections at first time step from PyPSA ---

t0 = net.snapshots[0]
print(f"First snapshot: {t0}\n")

# Generation at t0
gen_by_bus = net.generators_t.p.loc[t0].groupby(net.generators.bus).sum()

# Storage units dispatch at t0 (positive = discharging, negative = charging)
if not net.storage_units.empty and t0 in net.storage_units_t.p.index:
    stor_by_bus = net.storage_units_t.p.loc[t0].groupby(net.storage_units.bus).sum()
else:
    stor_by_bus = pd.Series(0.0, index=bus_order)

# Demand at t0
load_by_bus = net.loads_t.p_set.loc[t0].groupby(net.loads.bus).sum()

# Nodal injection = generation + storage dispatch - demand
p_full = pd.Series(0.0, index=bus_order)
for bus in bus_order:
    p_full[bus] = (gen_by_bus.get(bus, 0.0)
                 + stor_by_bus.get(bus, 0.0)
                 - load_by_bus.get(bus, 0.0))

print("Full injection vector [MW]:")
print(f"{'Bus':<12} {'Gen':>10} {'Storage':>10} {'Load':>10} {'Injection':>12}")
print("-" * 58)
for bus in bus_order:
    print(f"{bus:<12} {gen_by_bus.get(bus,0):>10.2f} "
          f"{stor_by_bus.get(bus,0):>10.2f} "
          f"{load_by_bus.get(bus,0):>10.2f} "
          f"{p_full[bus]:>12.2f}")

print(f"\nSum of all injections (must be ≈ 0): {p_full.sum():.4f} MW")

# Reduced injection vector: remove slack bus
p_r = p_full.drop(bus_order[slack_idx]).values
print(f"\nReduced injection vector p_r (excluding Denmark) [MW]:")
for name, val in zip(non_slack, p_r):
    print(f"  {name:<10}: {val:>10.2f} MW")

First snapshot: 2015-01-01 00:00:00

Full injection vector [MW]:
Bus                 Gen    Storage       Load    Injection
----------------------------------------------------------
Denmark         3193.65      31.40    3210.98        14.07
Germany        45568.44       0.00   44546.00      1022.44
Sweden         16258.31       0.00   14845.00      1413.31
Norway         12848.54     172.65   15471.00     -2449.81

Sum of all injections (must be ≈ 0): -0.0000 MW

Reduced injection vector p_r (excluding Denmark) [MW]:
  Germany   :    1022.44 MW
  Sweden    :    1413.31 MW
  Norway    :   -2449.81 MW


In [15]:
# --- E5: Line flows f = PTDF · p_r and verification against PyPSA ---

# Compute flows manually via PTDF
f_ptdf = PTDF @ p_r

# Read flows directly from PyPSA at t0
f_pypsa = net.lines_t.p0.loc[t0, [
    "line_DK_DE", "line_DK_SE", "line_DK_NO", "line_SE_NO", "line_DE_SE"
]].values

# Comparison
print("=" * 55)
print(f"{'PTDF VERIFICATION — ' + str(t0):^55}")
print("=" * 55)
print(f"{'Line':<12} {'PTDF [MW]':>12} {'PyPSA [MW]':>12} {'Δ [MW]':>10}")
print("-" * 55)
for line, fp, fy in zip(line_order, f_ptdf, f_pypsa):
    print(f"{line:<12} {fp:>12.2f} {fy:>12.2f} {fp-fy:>10.6f}")
print("=" * 55)
print(f"Max absolute difference: {abs(f_ptdf - f_pypsa).max():.6f} MW")

if abs(f_ptdf - f_pypsa).max() < 1e-3:
    print("✓ PTDF flows exactly reproduce PyPSA DC power flow at t0.")
else:
    print("✗ Discrepancy detected — check line ordering or injection vector.")

        PTDF VERIFICATION — 2015-01-01 00:00:00        
Line            PTDF [MW]   PyPSA [MW]     Δ [MW]
-------------------------------------------------------
DK-DE             -686.12      -686.12   0.000000
DK-SE             -349.81      -349.81   0.000000
DK-NO             1050.00      1050.00   0.000000
SE-NO             1399.81      1399.81  -0.000000
DE-SE              336.31       336.31  -0.000000
Max absolute difference: 0.000000 MW
✓ PTDF flows exactly reproduce PyPSA DC power flow at t0.
